# Customer Support Hierarchical Multi-Agent Flow (CrewAI)

## Objective

This notebook converts the previous customer-support workflow from a **sequential multi-agent process** into a **hierarchical multi-agent process**.

In the sequential version, agents run one after another in a fixed order:

```text
Issue Identification -> Knowledge Search -> Solution Generation -> Escalation Logic
```

In this hierarchical version, a **Support Operations Manager** supervises the workflow. The manager coordinates specialist agents, reviews their outputs, and ensures the final answer is complete, helpful, and safe to send.

```text
Support Operations Manager
   |-- Issue Identification Specialist
   |-- Knowledge Base Specialist
   |-- Solution Generator
   |-- Escalation Coordinator
```

The manager acts like a team lead rather than just another step in the chain.

## Install Dependencies

Run this if the required packages are not already installed.

```python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_openai python-dotenv
```

In [4]:
# !pip install langchain-openai langchain tiktoken python-dotenv
# !pip install crewai crewai_tools

In [5]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from crewai import Agent, Task, Crew, Process

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Create Specialist Agents

These are the worker agents. Each one owns a specific part of the support workflow.

In [6]:
issue_classifier = Agent(
    role="Issue Identification Specialist",
    goal="Correctly interpret and classify customer issues by type, sentiment, urgency, and priority.",
    backstory="You are trained in customer support triage and can quickly identify what the customer needs.",
    llm="gpt-4o-mini",
    verbose=True,
)

knowledge_agent = Agent(
    role="Knowledge Base Specialist",
    goal="Find the most relevant support guidance for the classified issue.",
    backstory="You are an expert at matching customer issues to internal help articles, known fixes, and policy guidance.",
    llm="gpt-4o-mini",
    verbose=True,
)

solution_agent = Agent(
    role="Solution Generator",
    goal="Write a clear, empathetic, customer-ready support response.",
    backstory="You write concise, warm support replies with practical next steps.",
    llm="gpt-4o-mini",
    verbose=True,
)

escalation_agent = Agent(
    role="Escalation Coordinator",
    goal="Decide whether the customer issue should be handled by AI or escalated to a human support agent.",
    backstory="You evaluate urgency, customer sentiment, missing information, policy risk, and unresolved cases.",
    llm="gpt-4o-mini",
    verbose=True,
)

## Create Manager Agent

This is what makes the workflow hierarchical. The manager does not simply perform one narrow task. It supervises the support process, coordinates specialist work, and checks the final result.

In [7]:
support_manager = Agent(
    role="Support Operations Manager",
    goal="Supervise the customer support workflow and ensure the final answer is accurate, complete, empathetic, and escalation-aware.",
    backstory=(
        "You are a senior support team lead. You coordinate specialists, review their work, "
        "resolve gaps, and make the final decision about what should be sent to the customer."
    ),
    llm="gpt-4o-mini",
    verbose=True,
    allow_delegation=True,
)

## Define Tasks

The specialist tasks still exist, but the crew will be run using a hierarchical process with the manager supervising the work.

In [8]:
identify_issue = Task(
    description="""Analyze the customer message: "{customer_query}".

Identify:
- Issue type: billing, technical, login, refund, or general inquiry
- Sentiment: angry, neutral, confused, urgent, or satisfied
- Priority level: low, normal, or high
- Short summary of the customer need

Return valid JSON with keys: type, urgency, sentiment, priority, summary.""",
    agent=issue_classifier,
    expected_output="A valid JSON object describing the classified issue.",
)

fetch_knowledge = Task(
    description="""Using the classification and customer message "{customer_query}", find the closest matching support guidance.

If the answer is unknown or policy-sensitive, return: NO MATCH FOUND.

Include any assumptions you made.""",
    agent=knowledge_agent,
    expected_output="A relevant support solution, policy note, or NO MATCH FOUND.",
)

generate_solution_response = Task(
    description="""Create a customer-ready support reply for: "{customer_query}".

Use the issue classification and knowledge-base guidance. The response must:
- Acknowledge the customer's problem
- Use plain language
- Give clear next steps
- Include an apology when appropriate
- Avoid making promises that require human approval""",
    agent=solution_agent,
    expected_output="A polished support response ready to send to the customer.",
)

apply_escalation_logic = Task(
    description="""Review the customer query, classification, retrieved guidance, and draft response.

Decide whether the case is:
- resolved
- requires_escalation

Escalate if the issue is urgent, angry, policy-sensitive, high financial impact, security-related, or if the solution was NO MATCH FOUND.

Return the status with a short justification and any notes for the human support agent.""",
    agent=escalation_agent,
    expected_output="Resolution status, justification, and escalation notes if needed.",
)

## Manager Review Task

The manager produces the final supervised output. This is the key difference from the sequential notebook.

In [9]:
manager_review = Task(
    description="""As the Support Operations Manager, review all specialist work for customer query: "{customer_query}".

Produce the final support package with:
1. Issue classification summary
2. Recommended customer response
3. Final status: resolved or requires_escalation
4. Manager quality check notes

Make sure the final response is useful, empathetic, and does not overpromise.""",
    agent=support_manager,
    expected_output="A manager-approved final support package.",
)

## Create Hierarchical Crew

`Process.hierarchical` tells CrewAI to use a manager-led workflow. The `manager_agent` coordinates the specialists and supervises the final result.

In [10]:
hierarchical_support_crew = Crew(
    agents=[issue_classifier, knowledge_agent, solution_agent, escalation_agent],
    tasks=[
        identify_issue,
        fetch_knowledge,
        generate_solution_response,
        apply_escalation_logic,
        manager_review,
    ],
    manager_agent=support_manager,
    process=Process.hierarchical,
    verbose=True,
)

## Run the Hierarchical Workflow

In [11]:
customer_query = """I'm unable to log into my account even though I'm sure the password is correct. Can someone help?"""

result = await hierarchical_support_crew.kickoff_async(inputs={"customer_query": customer_query})

print("\n--- FINAL MANAGER-APPROVED SUPPORT PACKAGE ---\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5abac71b-2b15-4d3d-ac43-dcc009ae809e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Operations Manager                                                                              │
│                                                                                                                 │
│  Task: Analyze the customer message: "I'm unable to log into my account even though I'm sure the password is    │
│  correct. Can someone help?".                                                                                   │
│                                                                                                                 │
│  Identify:                                                                                                      │
│  - Issue type: billing, technical, login, refund, or general inquiry                                            │
│  - Sentiment: angry, neutral, confused, urgent, or satisfied                                                    │
│  - Priority level: low, normal, or high                                                                         │
│  - Short summary of the customer need                                                                           │
│                                                                                                                 │
│  Return valid JSON with keys: type, urgency, sentiment, priority, summary.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the customer message: "I'm unable to log into my account even though I'm sure the password is    │
│  correct. Can someone help?".                                                                                   │
│                                                                                                                 │
│  Identify:                                                                                                      │
│  - Issue type: billing, technical, login, refund, or general inquiry                                            │
│  - Sentiment: angry, neutral, confused, urgent, or satisfied                                                    │
│  - Priority level: low, normal, or high                                                                         │
│  - Short summary of the customer need                                                                           │
│                                                                                                                 │
│  Return valid JSON with keys: type, urgency, sentiment, priority, summary.                                      │
│  ID: e22d6e1c-f226-4348-af13-6fa03d945312                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Analyze the customer message to classify the issue, sentiment, priority level, and provide a   │
│  summary.', 'context': 'Customer message: "I\'m unable to log into my account even though I\'m sur...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Issue Identification Specialist                                                                         │
│                                                                                                                 │
│  Task: Analyze the customer message to classify the issue, sentiment, priority level, and provide a summary.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Operations Manager                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Final Support Package**                                                                                      │
│                                                                                                                 │
│  1. **Issue Classification Summary**:                                                                           │
│     - **Type**: Login                                                                                           │
│     - **Urgency**: High                                                                                         │
│     - **Sentiment**: Confused                                                                                   │
│     - **Priority**: High                                                                                        │
│     - **Summary**: The customer is unable to log into their account despite being confident that their          │
│  password is correct.                                                                                           │
│                                                                                                                 │
│  2. **Recommended Customer Response**:                                                                          │
│     Subject: We're Here to Help with Your Login Issue                                                           │
│                                                                                                                 │
│     Hi there,                                                                                                   │
│                                                                                                                 │
│     I'm sorry to hear that you're having trouble logging into your account, even though you're confident that   │
│  your password is correct. It can be frustrating to face this kind of issue, but I'm here to help you get this  │
│  sorted out.                                                                                                    │
│                                                                                                                 │
│     Here are some steps to try:                                                                                 │
│                                                                                                                 │
│     1. **Double-check your password**: Make sure there are no typos, and that the Caps Lock key is not on, as   │
│  passwords are case-sensitive.                                                                                  │
│                                                                                                                 │
│     2. **Reset your password**: If you're still unable to log in, you can reset your password by clicking on    │
│  the "Forgot Password?" link on the login page. Follow the instructions in the email you receive to create a    │
│  new password.                                                                                                  │
│                                                                                                                 │
│     3. **Browser issues**: Sometimes, browser settings can cause login problems. Try clearing your browser's    │
│  cache and cookies or using a different browser or devi

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: As the Support Operations Manager, review all specialist work for customer query: "I'm unable to log     │
│  into my account even though I'm sure the password is correct. Can someone help?".                              │
│                                                                                                                 │
│  Produce the final support package with:                                                                        │
│  1. Issue classification summary                                                                                │
│  2. Recommended customer response                                                                               │
│  3. Final status: resolved or requires_escalation                                                               │
│  4. Manager quality check notes                                                                                 │
│                                                                                                                 │
│  Make sure the final response is useful, empathetic, and does not overpromise.                                  │
│  Agent: Support Operations Manager                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5abac71b-2b15-4d3d-ac43-dcc009ae809e                                                                       │
│  Final Output: **Final Support Package**                                                                        │
│                                                                                                                 │
│  1. **Issue Classification Summary**:                                                                           │
│     - **Type**: Login                                                                                           │
│     - **Urgency**: High                                                                                         │
│     - **Sentiment**: Confused                                                                                   │
│     - **Priority**: High                                                                                        │
│     - **Summary**: The customer is unable to log into their account despite being confident that their          │
│  password is correct.                                                                                           │
│                                                                                                                 │
│  2. **Recommended Customer Response**:                                                                          │
│     Subject: We're Here to Help with Your Login Issue                                                           │
│                                                                                                                 │
│     Hi there,                                                                                                   │
│                                                                                                                 │
│     I'm sorry to hear that you're having trouble logging into your account, even though you're confident that   │
│  your password is correct. It can be frustrating to face this kind of issue, but I'm here to help you get this  │
│  sorted out.                                                                                                    │
│                                                                                                                 │
│     Here are some steps to try:                                                                                 │
│                                                                                                                 │
│     1. **Double-check your password**: Make sure there are no typos, and that the Caps Lock key is not on, as   │
│  passwords are case-sensitive.                                                                                  │
│                                                                                                                 │
│     2. **Reset your password**: If you're still unable to log in, you can reset your password by clicking on    │
│  the "Forgot Password?" link on the login page. Follow the instructions in the email you receive to create a    │
│  new password.                                                                                                  │
│                                                                                                                 │
│     3. **Browser issues**: Sometimes, browser settings can cause login problems. Try clearing your browser's    │
│  cache and cookies or using a different browser or dev


--- FINAL MANAGER-APPROVED SUPPORT PACKAGE ---

**Final Support Package**

1. **Issue Classification Summary**: 
   - **Type**: Login
   - **Urgency**: High
   - **Sentiment**: Confused
   - **Priority**: High
   - **Summary**: The customer is unable to log into their account despite being confident that their password is correct.

2. **Recommended Customer Response**: 
   Subject: We're Here to Help with Your Login Issue

   Hi there,

   I'm sorry to hear that you're having trouble logging into your account, even though you're confident that your password is correct. It can be frustrating to face this kind of issue, but I'm here to help you get this sorted out.

   Here are some steps to try:

   1. **Double-check your password**: Make sure there are no typos, and that the Caps Lock key is not on, as passwords are case-sensitive.
   
   2. **Reset your password**: If you're still unable to log in, you can reset your password by clicking on the "Forgot Password?" link on the login pa

## Try More Queries

Replace `customer_query` with any of these examples:

```text
This is urgent. My business is down because your system won't sync orders. Fix this ASAP.

I was charged twice for my subscription this month. Please refund the extra amount.

The app keeps crashing every time I try to open the dashboard. This started after the latest update.

Does your software support exporting data to Excel? I can't find the option anywhere.
```